# `dash_interact` — Jupyter demo

The `page` object brings a **Streamlit-like authoring experience** to
Plotly Dash — and it works directly in Jupyter notebooks.

| | Streamlit | `dash_interact` |
|---|---|---|
| Layout | Top-to-bottom script | Top-to-bottom cells |
| Controls | `st.slider()`, `st.text_input()`, … | Derived from type hints |
| Display | `st.plotly_chart(fig)` | Return value of function |
| Launch | `streamlit run app.py` | Last expression in cell |
| Backend | Own server | Full Plotly Dash |

No `run()` call needed — placing a panel as the **last expression in a cell**
launches the app inline automatically.

## Setup

In [1]:
from typing import Literal

import numpy as np
import plotly.graph_objects as go
from dash_interact import interact, page

## Panel 1 — sine wave

Just decorate a typed function with `@interact` — no trailing expression, no
`run()` call.  When the cell finishes, the page auto-displays (the same
mechanism matplotlib uses to show figures after each cell).

In [2]:
@interact(amplitude=(0.1, 3.0, 0.1), frequency=(0.5, 8.0, 0.5))
def sine_wave(
    amplitude: float = 1.0,
    frequency: float = 2.0,
    ncycles: int = 3,
    color: Literal["royalblue", "crimson", "seagreen", "darkorange"] = "royalblue",
    show_envelope: bool = False,
):
    """Sine wave — adjust sliders to update the chart live."""
    x = np.linspace(0, ncycles * 2 * np.pi, 600)
    y = amplitude * np.sin(frequency * x)
    traces = [go.Scatter(x=x, y=y, mode="lines", line={"color": color, "width": 2})]
    if show_envelope:
        traces.extend(
            go.Scatter(
                x=x,
                y=[s * amplitude] * len(x),
                mode="lines",
                line={"color": color, "dash": "dot", "width": 1},
                showlegend=False,
            )
            for s in (1, -1)
        )
    return go.Figure(
        traces,
        layout={"margin": {"t": 20, "b": 40}, "yaxis": {"range": [-3.5, 3.5]}},
    )

: 

## Panel 2 — histogram

Each cell is independent — after the previous cell displayed, the page reset
automatically.  This cell builds a fresh page with just the histogram.

In [4]:
@interact(n_samples=(50, 2000, 50), bins=(5, 100, 5))
def histogram(
    n_samples: int = 500,
    mean: float = 0.0,
    std: float = 1.0,
    bins: int = 40,
    color: Literal["royalblue", "crimson", "seagreen", "darkorange"] = "seagreen",
):
    """Normal-distribution histogram."""
    rng = np.random.default_rng(seed=42)
    data = rng.normal(mean, max(std, 0.01), n_samples)
    return go.Figure(
        go.Histogram(x=data, nbinsx=bins, marker_color=color, opacity=0.8),
        layout={"margin": {"t": 20, "b": 40}, "bargap": 0.05},
    )

/Users/simonniederberger/Library/CloudStorage/Dropbox/Git/dash-fn-tools/packages/dash-fn-interact/src/dash_fn_interact/_forms.py:976: UserWarning: dash-fn-interact: config_id 'histogram' is already in use. Duplicate IDs will cause Dash callback errors. Pass _replace=True to suppress this warning.
  super().__init__(


## Accessing the page explicitly

`page()` returns the current singleton — useful for inspecting state
or to pass `jupyter_mode="external"` when you need more screen space.

In [ ]:
p = page()

print(f"Page has {len(p.children)} components")

# Open in a separate browser tab instead of inline:
# page().run(jupyter_mode="external", port=8070)